In [2]:
import pandas as pd
import os

# =============================
# 1. 参数设置
# =============================
INPUT_FILE = "/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.3.cNMF_NK/Example_refbuilder_NK/NK_ALL_cGEP_top_genes_long_fit.csv"
OUT_DIR = "/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.3.cNMF_NK/3.cGEP_Anno_NK/"
OUT_FILENAME = "3.NK_cGEP_Annotation_Results.csv" # 指定具体文件名

# 确保输出目录存在
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
OUT_PATH = os.path.join(OUT_DIR, OUT_FILENAME)

# =============================
# 2. 定义参考集 (直接使用字典)
# =============================
# 这是一个标准的 Python 字典，不需要再进行字符串解析
marker_dict = {
    # --- 亚群分类 (Subtypes) ---
    'NK_Immature_CD56bright': {'NCAM1', 'KLRC1', 'XCL1', 'XCL2', 'IL7R', 'CCR7'},
    'NK_Mature_CD56dim': {'FCGR3A', 'PRF1', 'GZMB', 'FGFBP2', 'CTSW'},
    'NK_Adaptive_Memory': {'KLRC2', 'B3GAT1', 'FCGR3A'},
    'Tissue_Resident_trNK': {'CD160', 'NR4A1', 'RGS1', 'CD69', 'CXCR6', 'ITGAE'},
    'TaNK_Stress': {'DNAJB1', 'HSPA1A', 'HSPA1B', 'NR4A2'},
    
    # --- 功能状态 (Functional States) ---
    'NK_Cytotoxic': {'PRF1', 'GZMB', 'GZMH', 'GNLY', 'NKG7', 'CTSW'},
    'NK_Activated': {'IFNG', 'TNF', 'CCL3', 'CCL4', 'NFKBIA', 'FOS', 'JUN'},
    'NK_Interferon': {'ISG15', 'IFIT1', 'IFIT3', 'MX1', 'OAS1', 'STAT1'},
    'NK_Cycling': {'MKI67', 'TOP2A', 'HMGB2', 'TUBA1B', 'TYMS'},
    'NK_Inhibitory': {'KLRC1', 'KLRD1', 'KIR2DL1', 'KIR2DL3'},
    'NK_Dysfunction': {'TIGIT', 'HAVCR2', 'LAG3', 'PDCD1'},
    
    # --- 质控与双胞剔除 (QC & Doublets) ---
    'Doublet_NK_T': {'CD3D', 'CD3E', 'CD3G', 'TRBC1', 'TRAC'},
    'Doublet_NK_Myeloid': {'LYZ', 'S100A8', 'S100A9', 'CTSD', 'LGALS3'},
    'Doublet_NK_B': {'MS4A1', 'CD79A', 'CD79B'},
    'Artifact_RBC': {'HBB', 'HBA1', 'HBA2'}
}

print(f"✅ 成功加载 {len(marker_dict)} 个参考类型。")

# =============================
# 3. 读取数据
# =============================
print(f"正在读取文件: {INPUT_FILE}")
# 注意：这里假设您的 CSV 是 宽数据格式 (每一列是一个 cGEP，列名是 cGEP 名称)
# 如果您的 CSV 是长数据格式 (包含 gene, cluster 列)，需要先用 pivot 转换
df = pd.read_csv(INPUT_FILE)

# =============================
# 4. 注释函数
# =============================
def annotate_cgep_spectra(gene_list):
    input_genes = set(gene_list)
    
    best_results = {
        "Spectra_Label": "Unknown",
        "Overlap_Count": 0,
        "Jaccard_Score": 0.0,
        "Key_Markers": ""
    }
    
    max_score = -1
    
    for ref_name, ref_genes in marker_dict.items():
        # 计算交集
        intersection = input_genes & ref_genes
        if not intersection:
            continue
            
        overlap_n = len(intersection)
        
        # --- 评分算法 ---
        # 1. Jaccard: 交集 / 并集 (适合衡量整体相似度)
        jaccard = overlap_n / len(input_genes | ref_genes)
        
        # 2. (可选) Overlap Coefficient: 交集 / 较小的集合
        # 这种算法在参考集很小(比如只有3个基因)但完全命中时，分数会很高
        # overlap_coef = overlap_n / min(len(input_genes), len(ref_genes))
        
        # 这里我们主要使用 Jaccard 作为排序依据
        score = jaccard
        
        if score > max_score:
            max_score = score
            best_results = {
                "Spectra_Label": ref_name,
                "Overlap_Count": overlap_n,
                "Jaccard_Score": round(score, 4),
                "Key_Markers": ",".join(list(intersection)) # 显示所有命中的 Marker
            }
            
    return best_results

# =============================
# 5. 执行注释循环
# =============================
results = []

# 遍历每一列 (假设列名是 cGEP 名称)
for cgep_col in df.columns:
    # 1. 获取当前 cGEP 的基因，去除空值，转为字符串并大写
    genes = (
        df[cgep_col]
        .dropna()
        .astype(str)
        .str.upper() # 确保大小写匹配
        .tolist()
    )
    
    # 2. 如果该列没有基因，跳过
    if not genes:
        continue

    # 3. 运行注释
    anno = annotate_cgep_spectra(genes)
    
    # 4. 收集结果
    results.append({
        "cGEP_Name": cgep_col,
        "Spectra_Label": anno["Spectra_Label"],
        "Overlap_Count": anno["Overlap_Count"],
        "Jaccard_Score": anno["Jaccard_Score"],
        "Key_Markers": anno["Key_Markers"],
        "Top_5_Genes_Input": ",".join(genes[:5]) # 方便核对
    })

# =============================
# 6. 输出结果
# =============================
out_df = pd.DataFrame(results)

# 按照分数降序排列，方便看匹配最好的
out_df = out_df.sort_values(by="Jaccard_Score", ascending=False)

# 保存
out_df.to_csv(OUT_PATH, index=False)

print("-" * 30)
print("✅ 注释完成！")
print(f"📄 结果已保存至: {OUT_PATH}")
print("-" * 30)
print("预览前 5 行结果:")
print(out_df[["cGEP_Name", "Spectra_Label", "Overlap_Count", "Jaccard_Score"]].head())

✅ 成功加载 15 个参考类型。
正在读取文件: /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.3.cNMF_NK/Example_refbuilder_NK/NK_ALL_cGEP_top_genes_long_fit.csv
------------------------------
✅ 注释完成！
📄 结果已保存至: /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.3.cNMF_NK/3.cGEP_Anno_NK/3.NK_cGEP_Annotation_Results.csv
------------------------------
预览前 5 行结果:
   cGEP_Name           Spectra_Label  Overlap_Count  Jaccard_Score
9     cGEP10              NK_Cycling              5         0.1667
13    cGEP14           NK_Interferon              5         0.1613
39    cGEP40  NK_Immature_CD56bright              5         0.1613
10    cGEP11            Doublet_NK_T              4         0.1290
1      cGEP2       NK_Mature_CD56dim              4         0.1290


In [3]:
out_df

,cGEP_Name,Spectra_Label,Overlap_Count,Jaccard_Score,Key_Markers,Top_5_Genes_Input
9,cGEP10,NK_Cycling,5,0.1667,"TUBA1B,TYMS,TOP2A,MKI67,HMGB2","TYMS,STMN1,TUBA1B,UBE2C,SPC25"
13,cGEP14,NK_Interferon,5,0.1613,"IFIT3,ISG15,OAS1,MX1,IFIT1","ISG15,IFIT3,IFI6,MX1,RSAD2"
39,cGEP40,NK_Immature_CD56bright,5,0.1613,"KLRC1,XCL2,NCAM1,XCL1,IL7R","XCL1,GPR183,CD44,SELL,TCF7"
10,cGEP11,Doublet_NK_T,4,0.1290,"CD3E,CD3G,TRAC,CD3D","CD3D,CD3G,SIT1,CD8B,S100A6"
1,cGEP2,NK_Mature_CD56dim,4,0.1290,"FGFBP2,FCGR3A,PRF1,GZMB","NKG7,PRF1,SPON2,FGFBP2,GZMB"
31,cGEP32,Artifact_RBC,3,0.1000,"HBB,HBA2,HBA1","HBB,HBA2,ALAS2,SLC25A37,SNCA"
33,cGEP34,NK_Dysfunction,3,0.0968,"PDCD1,TIGIT,LAG3","LOC101928173,MIR155HG,TNFRSF9,CD82,MT1L"
4,cGEP5,TaNK_Stress,3,0.0968,"DNAJB1,HSPA1A,HSPA1B","DNAJB1,HSPA1A,HSP90AA1,HSPE1,HSPA1B"
2,cGEP3,TaNK_Stress,3,0.0968,"DNAJB1,HSPA1A,HSPA1B","PFN1,ACTB,FCER1G,HSPA1A,IFITM1"
23,cGEP24,Doublet_NK_T,3,0.0938,"CD3E,CD3G,CD3D","CD3D,CD8B,CD8A,PTPRC,HLA-A"
